In [32]:
!pip install -q ddgs ollama
!apt-get install -y zstd -qq
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q statsmodels


!pkill -f "ollama serve"
import time
time.sleep(2)

import subprocess, time

with open('/kaggle/working/ollama.log', 'w') as log_file:
    subprocess.Popen(['ollama', 'serve'], stdout=log_file, stderr=log_file)

time.sleep(5)
print("Service Ollama demarre")

!ollama pull gemma3:4b
!ollama pull qwen2.5:3b
!ollama pull llama3.2:3b
!ollama pull phi4-mini

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%####################                                      51.4%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Service Ollama demarre
]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest 
pulling aeda25e63ebd: 100% ▕██████████████████▏ 3.3 GB                         
pulling e0a42594d802: 100% ▕██████████████████▏  358 B                         
pulling dd084c7d92a3: 100% ▕██████████████████▏ 8.4 KB                         
pulling 3116c5225075: 100% ▕██████████████████▏   77 B                         
pulling b6ae5839783f: 100% ▕██████████████████▏  489 B        

In [33]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import csv
import random
import re
import time
import ollama
from ddgs import DDGS
from collections import Counter
from statsmodels.stats.contingency_tables import cochrans_q
from scipy.stats import chi2
import numpy as np

In [34]:
!nvidia-smi

Tue Oct  6 04:45:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [35]:
def lire_csv(chemin):
    with open(chemin, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))

In [36]:
chemin_in = "/kaggle/input/datasets/imenec/data-n/"
chemin_out = "/kaggle/working/"

articles_B = lire_csv(chemin_in + "cat_B_150.csv")
articles_D = lire_csv(chemin_in + "cat_D_150.csv")

print(f"Catégorie B : {len(articles_B)} articles")
print(f"Catégorie D : {len(articles_D)} articles")

Catégorie B : 150 articles
Catégorie D : 150 articles


In [37]:
def chercher_preuves(titre, texte):
    # cherche sur web des infos sur le texte passé en param
    # retourne les 3 premiers resultats trouvés formatés en liste de points

    requete = titre if titre else texte[:100]
    try:
        resultats = DDGS().text(requete, max_results=3)
    except Exception as e:
        print(f"  Erreur de recherche : {e}")
        return "(aucune preuve trouvee)"

    if not resultats:
        return "(aucune preuve trouvee)"

    morceaux = []
    for r in resultats:
        morceaux.append(f"- {r.get('title', '')} : {r.get('body', '')[:200]}")
    return "\n".join(morceaux)


In [7]:
def chercher_preuves_2(titre, texte, max_tentatives=3):
    # cherche sur web des infos sur le texte passé en param
    # retourne les 3 premiers resultats trouvés formatés en liste de points
    # difference avec l'autre, si on échoue a trouver des preuves, on essaye jusqu'a 3 tentatives

    requete = titre if titre else texte[:100]

    for tentative in range(1, max_tentatives + 1):
        try:
            # timeout augmente a 20s (au lieu de 5s par defaut)
            with DDGS(timeout=20) as ddgs:
                resultats = ddgs.text(query=requete, max_results=3)

            if not resultats:
                return "(aucune preuve trouvee)"

            morceaux = []
            for r in resultats:
                morceaux.append(f"- {r.get('title', '')} : {r.get('body', '')[:200]}")
            return "\n".join(morceaux)

        except Exception as e:
            print(f"  [!] Tentative {tentative}/{max_tentatives} echouee : {e}")
            if tentative < max_tentatives:
                # attend de plus en plus longtemps a chaque tentative (3s, 6s, 9s)
                time.sleep(3 * tentative)

    # si toutes les tentatives ont echoue
    return "(aucune preuve trouvee apres plusieurs tentatives)"


In [41]:
def construire_prompt(titre, texte, source, preuves, avec_rag):
    texte_court = texte[:800]

    prompt = "Voici un article a verifier.\n\n"
    prompt = prompt + "Source : " + source + "\n"
    prompt = prompt + "Titre : " + titre + "\n"
    prompt = prompt + "Texte : " + texte_court + "\n\n"

    if avec_rag:
        prompt = prompt + "Preuves trouvees sur le web concernant cette affirmation :\n"
        prompt = prompt + preuves + "\n\n"
        prompt = prompt + "En te basant sur ces preuves, verifie si l'affirmation de cet article est vraie ou fausse.\n"
    else:
        prompt = prompt + "En te basant sur ce que tu connais déja, verifie si l'affirmation de cet article est vraie ou fausse.\n"
    
    prompt = prompt + "Reponds uniquement par un seul mot : \"fake\" ou \"real\"."

    return prompt


def extraire_verdict(reponse_llm):
    texte = reponse_llm.lower()
    if re.search(r"\bfake\b", texte):
        return "fake"
    if re.search(r"\breal\b", texte):
        return "real"
    return "indetermine"


def extraire_verdict_2(reponse_llm):
    texte = reponse_llm.lower().strip()
    # Si la reponse est longue, c'est surement pas un real ou un fake 
    if len(texte.split()) > 5:
        return "indetermine"
    if re.search(r"\bfake\b", texte):
        return "fake"
    if re.search(r"\breal\b", texte):
        return "real"
    return "indetermine"

In [43]:
def demander_verdict(titre, texte, source, preuves, avec_rag):

    prompt = construire_prompt(titre, texte, source, preuves, avec_rag)

    reponse = ollama.chat(model=m, messages=[{"role": "user", "content": prompt}])

    contenu = reponse["message"]["content"]

    return extraire_verdict(contenu)


In [44]:
def verdict_n_fois(titre, texte, source, preuves, avec_rag, n=5):
    # Fonction qui appelle le LLM n fois avec les memes parametres
    # et renvoie le verdict 
    # on lance n fois pour chercher une certaine stabilité dans les reponses du modele
    # apres je prends la verdict majoritaire retourné

    verdicts = []
    for i in range(n):
        v = demander_verdict(titre, texte, source, preuves, avec_rag)
        verdicts.append(v)
        
    # compte combien a chaque fois cahque valeur apparait dans la liste
    compte = Counter(verdicts)
    # retourne l'element le plus frequent, [0][0] pour chercher l'etiquette real ou fake
    majorite = compte.most_common(1)[0][0]
    return majorite, verdicts

In [45]:
def tester_contrefactuel(article, source_2, avec_rag, creer_preuves, pas_source="source non identifiée"):
    
    # exemple de source peu connue : yournewswire.com
    # exemple de source connue : dailymail.co.uk
    # Je teste un article 3 fois : 
    # une fois avec sa vraie source originale,
    # une fois avec une source oposée au statut de la source originale
    # si la source originale est connue, on changera pour une source peu_connue et virce versa
    # une fois en anonymisant la source en mettant a la place "source non identifiée"
    # mais en gardant le meme contenu, meme preuves.
    
    titre = article.get("title", "")
    texte = article.get("text", "")
    vraie_source = article.get("domaine", "source inconnue")
    vrai_label = article.get("label", "")

    # RAG ou non-RAG
    if avec_rag:
        if creer_preuves:
            # la 1ere config de Gemma cherche les preuves une seule fois
            preuves = chercher_preuves(titre, texte)
            
            # on sauvegarde les preuves directement dans l'article
            article["preuves"] = preuves
            print(f"  RAG CRÉÉ pour : {article.get('id', '')}")
        else:
            # les autres modele vont reutiliser les preuves de Gemma
            preuves = article["preuves"]
            print(f"  RAG RÉUTILISÉ pour : {article.get('id', '')}")
    else:
        preuves = None


    # Version 1 : avec la vraie source 
    verdict_original, n_original = verdict_n_fois(titre, texte, vraie_source, preuves, avec_rag)

    # Version 2 : source changée pour l'opposé de la source originale
    verdict_oppose, n_oppose = verdict_n_fois(titre, texte, source_2, preuves, avec_rag)

    # Version 3 : source rendue anonyme
    verdict_anonyme, n_anonyme = verdict_n_fois(titre, texte, pas_source, preuves, avec_rag)
    

    # flip = oui sil y a au moins un verdict different entre les 3 rendus
    if verdict_original == verdict_oppose == verdict_anonyme:
        flip = "non"
    else:
        flip = "oui"

    return {
        "id": article.get("id", ""),
        "vraie_source": vraie_source,
        "vrai_label": vrai_label,
        "verdict_original": verdict_original,
        "verdict_oppose": verdict_oppose,
        "verdict_anonyme": verdict_anonyme,
        "flip": flip,
    }

In [46]:
def lancer_test_contrefactuel(echantillon, chemin_sortie, source_2, avec_rag, creer_preuves):

    colonnes = ["id", "vraie_source", "vrai_label", "verdict_original", "verdict_oppose" ,"verdict_anonyme", "flip"]
    resultats = []

    with open(chemin_sortie, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=colonnes)
        writer.writeheader()

        for i, article in enumerate(echantillon):
            r = tester_contrefactuel(article, source_2, avec_rag, creer_preuves)
            writer.writerow(r)
            resultats.append(r)
            print(f"[{i+1}/{len(echantillon)}] {r['vraie_source']} -> vraiLabel={r['vrai_label']}, original={r['verdict_original']}, oppose={r['verdict_oppose']} ,anonyme={r['verdict_anonyme']}, flip={r['flip']}")
            
    return resultats

In [47]:
# GEMMA
m = "gemma3:4b"

print("================ GEMMA - catégorie B - avec RAG =====================")
resultats_rag = lancer_test_contrefactuel(
    echantillon=articles_B,
    chemin_sortie=chemin_out + "gemma_B_RAG.csv",
    source_2="yournewswire.com", 
    avec_rag=True,
    creer_preuves=True
)

print("================ GEMMA - catégorie B - sans RAG =====================")
resultats_WRAG = lancer_test_contrefactuel(
    echantillon=articles_B,
    chemin_sortie=chemin_out + "gemma_B_WRAG.csv",
    source_2="yournewswire.com", 
    avec_rag=False,
    creer_preuves=False
)


# QWEN
m = "qwen2.5:3b"

print("================ QWEN - catégorie B - avec RAG =====================")
resultats_rag = lancer_test_contrefactuel(
    echantillon=articles_B,
    chemin_sortie=chemin_out + "qwen_B_RAG.csv",
    source_2="yournewswire.com", 
    avec_rag=True,
    creer_preuves=False
)

print("================ QWEN - catégorie B - sans RAG =====================")
resultats_WRAG = lancer_test_contrefactuel(
    echantillon=articles_B,
    chemin_sortie=chemin_out + "qwen_B_WRAG.csv",
    source_2="yournewswire.com", 
    avec_rag=False,
    creer_preuves=False
)


#llama
m = "llama3.2:3b"

print("================ LLAMA - catégorie B - avec RAG =====================")
lancer_test_contrefactuel(
    echantillon=articles_B,
    chemin_sortie=chemin_out + "llama_B_RAG.csv",
    source_2="yournewswire.com",
    avec_rag=True,
    creer_preuves=False
)

print("================ LLAMA - catégorie B - sans RAG =====================")
lancer_test_contrefactuel(
    echantillon=articles_B,
    chemin_sortie=chemin_out + "llama_B_WRAG.csv",
    source_2="yournewswire.com",
    avec_rag=False,
    creer_preuves=False
)



#phi4
m = "phi4-mini"

print("================ PHI4 - catégorie B - avec RAG =====================")
lancer_test_contrefactuel(
    echantillon=articles_B,
    chemin_sortie=chemin_out + "phi4_B_RAG.csv",
    source_2="yournewswire.com",
    avec_rag=True,
    creer_preuves=False
)

print("================ PHI4 - catégorie B - sans RAG =====================")
lancer_test_contrefactuel(
    echantillon=articles_B,
    chemin_sortie=chemin_out + "phi4_B_WRAG.csv",
    source_2="yournewswire.com",
    avec_rag=False,
    creer_preuves=False
)

================ GEMMA - catégorie B - avec RAG =====================
  RAG CRÉÉ pour : gossipcop-921023
[1/150] usmagazine.com -> vraiLabel=real, original=real, oppose=real ,anonyme=real, flip=non
  Erreur de recherche : No results found.
  RAG CRÉÉ pour : gossipcop-862461
[2/150] tmz.com -> vraiLabel=real, original=real, oppose=fake ,anonyme=fake, flip=oui
  RAG CRÉÉ pour : gossipcop-899258
[3/150] popculture.com -> vraiLabel=real, original=real, oppose=real ,anonyme=real, flip=non
  Erreur de recherche : No results found.
  RAG CRÉÉ pour : gossipcop-896446
[4/150] express.co.uk -> vraiLabel=real, original=fake, oppose=fake ,anonyme=fake, flip=non
  RAG CRÉÉ pour : gossipcop-906443
[5/150] today.com -> vraiLabel=real, original=real, oppose=real ,anonyme=real, flip=non
  Erreur de recherche : No results found.
  RAG CRÉÉ pour : gossipcop-842391
[6/150] tvline.com -> vraiLabel=real, original=real, oppose=real ,anonyme=real, flip=non
  Erreur de recherche : No results found.
  RAG CRÉÉ 

h2 connection driver error: connection reset


  Erreur de recherche : DecodeError: DecodeError('error decoding response body > request or response body error > error sending request > connection reset', None)
  RAG CRÉÉ pour : gossipcop-944289
[116/150] mashable.com -> vraiLabel=real, original=real, oppose=fake ,anonyme=fake, flip=oui
  RAG CRÉÉ pour : gossipcop-878994
[117/150] dailymail.co.uk -> vraiLabel=real, original=real, oppose=real ,anonyme=real, flip=non
  RAG CRÉÉ pour : gossipcop-862473
[118/150] today.com -> vraiLabel=real, original=real, oppose=real ,anonyme=real, flip=non
  RAG CRÉÉ pour : gossipcop-873980
[119/150] refinery29.com -> vraiLabel=real, original=real, oppose=real ,anonyme=real, flip=non
  RAG CRÉÉ pour : gossipcop-6751025589
[120/150] helenastales.weebly.com -> vraiLabel=fake, original=fake, oppose=fake ,anonyme=fake, flip=non
  RAG CRÉÉ pour : gossipcop-865272
[121/150] en.vogue.fr -> vraiLabel=real, original=real, oppose=real ,anonyme=real, flip=non
  RAG CRÉÉ pour : gossipcop-896859
[122/150] aljazeer

[{'id': 'gossipcop-921023',
  'vraie_source': 'usmagazine.com',
  'vrai_label': 'real',
  'verdict_original': 'real',
  'verdict_oppose': 'real',
  'verdict_anonyme': 'real',
  'flip': 'non'},
 {'id': 'gossipcop-862461',
  'vraie_source': 'tmz.com',
  'vrai_label': 'real',
  'verdict_original': 'real',
  'verdict_oppose': 'real',
  'verdict_anonyme': 'fake',
  'flip': 'oui'},
 {'id': 'gossipcop-899258',
  'vraie_source': 'popculture.com',
  'vrai_label': 'real',
  'verdict_original': 'real',
  'verdict_oppose': 'real',
  'verdict_anonyme': 'real',
  'flip': 'non'},
 {'id': 'gossipcop-896446',
  'vraie_source': 'express.co.uk',
  'vrai_label': 'real',
  'verdict_original': 'real',
  'verdict_oppose': 'real',
  'verdict_anonyme': 'real',
  'flip': 'non'},
 {'id': 'gossipcop-906443',
  'vraie_source': 'today.com',
  'vrai_label': 'real',
  'verdict_original': 'real',
  'verdict_oppose': 'fake',
  'verdict_anonyme': 'fake',
  'flip': 'oui'},
 {'id': 'gossipcop-842391',
  'vraie_source': 't

In [48]:
# GEMMA
m = "gemma3:4b"

print("================ GEMMA - catégorie D - avec RAG =====================")
resultats_rag = lancer_test_contrefactuel(
    echantillon=articles_D,
    chemin_sortie=chemin_out + "gemma_D_RAG.csv",
    source_2="dailymail.co.uk", 
    avec_rag=True,
    creer_preuves=True
)

print("================ GEMMA - catégorie D - sans RAG =====================")
resultats_WRAG = lancer_test_contrefactuel(
    echantillon=articles_D,
    chemin_sortie=chemin_out + "gemma_D_WRAG.csv",
    source_2="dailymail.co.uk", 
    avec_rag=False,
    creer_preuves=False
)

# QWEN
m = "qwen2.5:3b"

print("================ QWEN - catégorie D - avec RAG =====================")
resultats_rag = lancer_test_contrefactuel(
    echantillon=articles_D,
    chemin_sortie=chemin_out + "qwen_D_RAG.csv",
    source_2="dailymail.co.uk", 
    avec_rag=True,
    creer_preuves=False
)

print("================ QWEN - catégorie D - sans RAG =====================")
resultats_WRAG = lancer_test_contrefactuel(
    echantillon=articles_D,
    chemin_sortie=chemin_out + "qwen_D_WRAG.csv",
    source_2="dailymail.co.uk", 
    avec_rag=False,
    creer_preuves=False
)


# LLAMA
m = "llama3.2:3b"

print("================ LLAMA - catégorie D - avec RAG =====================")
lancer_test_contrefactuel(
    echantillon=articles_D,
    chemin_sortie=chemin_out + "llama_D_RAG.csv",
    source_2="dailymail.co.uk",
    avec_rag=True,
    creer_preuves=False
)

print("================ LLAMA - catégorie D - sans RAG =====================")
lancer_test_contrefactuel(
    echantillon=articles_D,
    chemin_sortie=chemin_out + "llama_D_WRAG.csv",
    source_2="dailymail.co.uk",
    avec_rag=False,
    creer_preuves=False
)


# PHI-4 MINI
m = "phi4-mini"

print("================ PHI4 - catégorie D - avec RAG =====================")
lancer_test_contrefactuel(
    echantillon=articles_D,
    chemin_sortie=chemin_out + "phi4_D_RAG.csv",
    source_2="dailymail.co.uk",
    avec_rag=True,
    creer_preuves=False
)

print("================ PHI4 - catégorie D - sans RAG =====================")
lancer_test_contrefactuel(
    echantillon=articles_D,
    chemin_sortie=chemin_out + "phi4_D_WRAG.csv",
    source_2="dailymail.co.uk",
    avec_rag=False,
    creer_preuves=False
)


================ GEMMA - catégorie D - avec RAG =====================
  Erreur de recherche : error sending request for url (https://html.duckduckgo.com/html/) > operation timed out
  RAG CRÉÉ pour : gossipcop-945865
[1/150] remonews.com -> vraiLabel=real, original=real, oppose=real ,anonyme=fake, flip=oui
  Erreur de recherche : error sending request for url (https://html.duckduckgo.com/html/) > operation timed out
  RAG CRÉÉ pour : gossipcop-896215
[2/150] worldnews.easybranches.com -> vraiLabel=real, original=real, oppose=real ,anonyme=real, flip=non
  Erreur de recherche : No results found.
  RAG CRÉÉ pour : gossipcop-886987
[3/150] longroom.com -> vraiLabel=real, original=real, oppose=real ,anonyme=fake, flip=oui
  RAG CRÉÉ pour : gossipcop-7946844362
[4/150] newidea.com.au -> vraiLabel=fake, original=real, oppose=real ,anonyme=real, flip=non
  RAG CRÉÉ pour : gossipcop-9972962159
[5/150] thefashionfauxpasofgabrielle.com -> vraiLabel=fake, original=real, oppose=real ,anonyme=real,

[{'id': 'gossipcop-945865',
  'vraie_source': 'remonews.com',
  'vrai_label': 'real',
  'verdict_original': 'real',
  'verdict_oppose': 'real',
  'verdict_anonyme': 'real',
  'flip': 'non'},
 {'id': 'gossipcop-896215',
  'vraie_source': 'worldnews.easybranches.com',
  'vrai_label': 'real',
  'verdict_original': 'real',
  'verdict_oppose': 'real',
  'verdict_anonyme': 'real',
  'flip': 'non'},
 {'id': 'gossipcop-886987',
  'vraie_source': 'longroom.com',
  'vrai_label': 'real',
  'verdict_original': 'real',
  'verdict_oppose': 'real',
  'verdict_anonyme': 'real',
  'flip': 'non'},
 {'id': 'gossipcop-7946844362',
  'vraie_source': 'newidea.com.au',
  'vrai_label': 'fake',
  'verdict_original': 'real',
  'verdict_oppose': 'real',
  'verdict_anonyme': 'real',
  'flip': 'non'},
 {'id': 'gossipcop-9972962159',
  'vraie_source': 'thefashionfauxpasofgabrielle.com',
  'vrai_label': 'fake',
  'verdict_original': 'real',
  'verdict_oppose': 'real',
  'verdict_anonyme': 'real',
  'flip': 'non'},
 

In [ ]:
!ls -lh /kaggle/working/

**TEST COCHRAN**


In [5]:
def calcul_cochran_q(chemin_csv, nom, col_connu, col_peu_connu, col_anonyme):
    articles = lire_csv(chemin_csv)

    tab=[]
    for a in articles:
        if a[col_connu] == "real":
            connu = 1
        else:
            connu = 0

        if a[col_peu_connu] == "real":
            peu_connu = 1
        else:
            peu_connu = 0

        if a[col_anonyme] == "real":
            anonyme = 1
        else:
            anonyme = 0

        tab.append([connu, peu_connu, anonyme])
        
    resultat = cochrans_q(tab)

    print(f"=== {nom} ===")
    print(f"  Statistique Q : {resultat.statistic:.2f}")
    print(f"  p-value : {resultat.pvalue:.2e}")

    if resultat.pvalue < 0.001:
        print("  => Les 3 conditions different significativement entre elles (p<0.001)")
    else:
        print("  => Pas de difference significative globale")
    print()

    return resultat
    

In [6]:
chemin_in = "/kaggle/input/datasets/imenec/experiences-results/"

resultat_B_avecRAG = calcul_cochran_q(
    chemin_csv=chemin_in + "qwen_B_RAG.csv",
    nom="Categorie B (avec RAG)",
    col_connu="verdict_original",
    col_peu_connu="verdict_oppose",
    col_anonyme="verdict_anonyme"
)

resultat_D_avecRAG = calcul_cochran_q(
    chemin_csv=chemin_in + "qwen_D_RAG.csv",
    nom="Categorie D (avec RAG)",
    col_connu="verdict_oppose",
    col_peu_connu="verdict_original",
    col_anonyme="verdict_anonyme"
)

resultat_B_sansRAG = calcul_cochran_q(
    chemin_csv=chemin_in + "qwen_B_WRAG.csv",
    nom="Categorie B (sans RAG)",
    col_connu="verdict_original",
    col_peu_connu="verdict_oppose",
    col_anonyme="verdict_anonyme"
)

resultat_D_sansRAG = calcul_cochran_q(
    chemin_csv=chemin_in + "qwen_D_WRAG.csv",
    nom="Categorie D (sans RAG)",
    col_connu="verdict_oppose",
    col_peu_connu="verdict_original",
    col_anonyme="verdict_anonyme"
)

=== Categorie B (avec RAG) ===
  Statistique Q : 0.25
  p-value : 8.82e-01
  => Pas de difference significative globale

=== Categorie D (avec RAG) ===
  Statistique Q : 6.89
  p-value : 3.19e-02
  => Pas de difference significative globale

=== Categorie B (sans RAG) ===
  Statistique Q : 43.09
  p-value : 4.40e-10
  => Les 3 conditions different significativement entre elles (p<0.001)

=== Categorie D (sans RAG) ===
  Statistique Q : 45.56
  p-value : 1.28e-10
  => Les 3 conditions different significativement entre elles (p<0.001)



**Test mcneamar**

In [7]:
 
def comparer_deux_colonnes(chemin_csv, nom_colonne_1, nom_colonne_2):
    # cette fonction compare 2 colonnes dun fichier csv avec le test mcNeamar
    # la fonction retourne un dictionnaire avec b, c, p-value,
    # proportions "real" de chaque colonne, diff + IC95%
    # b: nb d'articles "real" (colonne 1) devenus "fake" (colonne 2)
    # c: nb d'articles "fake" (colonne 1) devenus "real" (colonne 2)
    # p_value: probabilite d'observer ce desequilibre entre b et c
    # p_real_colonne_1: proportion d'articles "real" dans la colonne 1
    # p_real_colonne_2: proportion d'articles "real" dans la colonne 2
    # diff_proportions: variation de la proportion de "real" (colonne_2 - colonne_1)
    # ic95_low/high:(95% de confiance) la plus petite et la plus grande valeur probable de la "vraie" difference
 
    col1, col2 = [], []
    with open(chemin_csv, newline="", encoding="utf-8") as f:
        for ligne in csv.DictReader(f):
            col1.append(ligne[nom_colonne_1].strip().lower())
            col2.append(ligne[nom_colonne_2].strip().lower())
 
    n = len(col1)
 
    table = {"fake_fake": 0, "fake_real": 0, "real_fake": 0, "real_real": 0}
    for a, b_ in zip(col1, col2):
        table[f"{a}_{b_}"] += 1
 
    b = table["real_fake"]   # col1=real, col2=fake 
    c = table["fake_real"]   # col1=fake, col2=real
    nbr_flip = b + c
 
    # mcnemar
    if nbr_flip == 0:
        stat, p = 0.0, 1.0
    else:
        stat = (abs(b - c) - 1) ** 2 / nbr_flip
        p = 1 - chi2.cdf(stat, df=1)
 
    # proportions + IC95% de la difference appariee
    p1 = col1.count("real") / n #proportion d'articles "real" dans la colonne 1
    p2 = col2.count("real") / n #proportion d'articles "real" dans la colonne 2
    
    diff = p2 - p1
    
    se = ((b + c) - (b - c) ** 2 / n) ** 0.5 / n
    ic_low, ic_high = diff - 1.96 * se, diff + 1.96 * se
 
    return {
        "colonne_1": nom_colonne_1,
        "colonne_2": nom_colonne_2,
        "b_real_to_fake": b,
        "c_fake_to_real": c,
        "p_value": p,
        "p_real_colonne_1": p1,
        "p_real_colonne_2": p2,
        "diff_proportions": diff,
        "ic95_low": ic_low,
        "ic95_high": ic_high,
    }

In [9]:
 
def afficher_resultat(res, alpha=0.05/3):
    #Afficher les resultats de comparer_deux_colonnes() de facon lisible
    
    print(f"Comparaison : {res['colonne_1']} vs {res['colonne_2']}")
    print(f"  b (real->fake) = {res['b_real_to_fake']}, c (fake->real) = {res['c_fake_to_real']}, "
          f"changements = {res['b_real_to_fake'] + res['c_fake_to_real']}")
    print(f"  McNemar : p = {res['p_value']:.4g} "
          f"-> {'significatif' if res['p_value'] < alpha else 'non significatif'} (alpha={alpha})")
    print(f"  Proportion 'real' : {res['p_real_colonne_1']*100:.1f}% -> {res['p_real_colonne_2']*100:.1f}%")
    print(f"  Difference = {res['diff_proportions']*100:+.1f} points, "
          f"IC95% = [{res['ic95_low']*100:+.1f} ; {res['ic95_high']*100:+.1f}]")
    print()
 

 

In [10]:
chemin_in = "/kaggle/input/datasets/imenec/experiences-results/gemma_B_RAG.csv"
print("Gemma sur catégorie B avec RAG")
# cat B , original = connu
# connu vs peu_conu
res1 = comparer_deux_colonnes(chemin_in, "verdict_original", "verdict_oppose")
afficher_resultat(res1)

# peu_connu vs anonyme
res2 = comparer_deux_colonnes(chemin_in, "verdict_oppose", "verdict_anonyme")
afficher_resultat(res2)

# connu vs anonyme
res3 = comparer_deux_colonnes(chemin_in, "verdict_original", "verdict_anonyme")
afficher_resultat(res3)

Gemma sur catégorie B avec RAG
Comparaison : verdict_original vs verdict_oppose
  b (real->fake) = 12, c (fake->real) = 1, changements = 13
  McNemar : p = 0.005546 -> significatif (alpha=0.016666666666666666)
  Proportion 'real' : 78.0% -> 70.7%
  Difference = -7.3 points, IC95% = [-11.9 ; -2.8]

Comparaison : verdict_oppose vs verdict_anonyme
  b (real->fake) = 4, c (fake->real) = 3, changements = 7
  McNemar : p = 1 -> non significatif (alpha=0.016666666666666666)
  Proportion 'real' : 70.7% -> 70.0%
  Difference = -0.7 points, IC95% = [-4.1 ; +2.8]

Comparaison : verdict_original vs verdict_anonyme
  b (real->fake) = 13, c (fake->real) = 1, changements = 14
  McNemar : p = 0.003283 -> significatif (alpha=0.016666666666666666)
  Proportion 'real' : 78.0% -> 70.0%
  Difference = -8.0 points, IC95% = [-12.7 ; -3.3]



In [11]:
chemin_in = "/kaggle/input/datasets/imenec/experiences-results/gemma_B_WRAG.csv"
print("Gemma sur catégorie B sans RAG")
# cat B , original = connu
# connu vs peu_conu
res1 = comparer_deux_colonnes(chemin_in, "verdict_original", "verdict_oppose")
afficher_resultat(res1)

# peu_connu vs anonyme
res2 = comparer_deux_colonnes(chemin_in, "verdict_oppose", "verdict_anonyme")
afficher_resultat(res2)

# connu vs anonyme
res3 = comparer_deux_colonnes(chemin_in, "verdict_original", "verdict_anonyme")
afficher_resultat(res3)

Gemma sur catégorie B sans RAG
Comparaison : verdict_original vs verdict_oppose
  b (real->fake) = 19, c (fake->real) = 1, changements = 20
  McNemar : p = 0.0001439 -> significatif (alpha=0.016666666666666666)
  Proportion 'real' : 78.7% -> 66.7%
  Difference = -12.0 points, IC95% = [-17.5 ; -6.5]

Comparaison : verdict_oppose vs verdict_anonyme
  b (real->fake) = 1, c (fake->real) = 13, changements = 14
  McNemar : p = 0.003283 -> significatif (alpha=0.016666666666666666)
  Proportion 'real' : 66.7% -> 74.7%
  Difference = +8.0 points, IC95% = [+3.3 ; +12.7]

Comparaison : verdict_original vs verdict_anonyme
  b (real->fake) = 8, c (fake->real) = 2, changements = 10
  McNemar : p = 0.1138 -> non significatif (alpha=0.016666666666666666)
  Proportion 'real' : 78.7% -> 74.7%
  Difference = -4.0 points, IC95% = [-8.1 ; +0.1]



In [12]:
chemin_in = "/kaggle/input/datasets/imenec/experiences-results/gemma_D_RAG.csv"
print("Gemma sur catégorie D avec RAG")
# cat D , original = peu connu
# connu vs peu_conu
res1 = comparer_deux_colonnes(chemin_in, "verdict_oppose", "verdict_original")
afficher_resultat(res1)

# peu_connu vs anonyme
res2 = comparer_deux_colonnes(chemin_in, "verdict_original", "verdict_anonyme")
afficher_resultat(res2)

# connu vs anonyme
res3 = comparer_deux_colonnes(chemin_in, "verdict_oppose", "verdict_anonyme")
afficher_resultat(res3)

Gemma sur catégorie D avec RAG
Comparaison : verdict_oppose vs verdict_original
  b (real->fake) = 8, c (fake->real) = 0, changements = 8
  McNemar : p = 0.01333 -> significatif (alpha=0.016666666666666666)
  Proportion 'real' : 62.0% -> 56.7%
  Difference = -5.3 points, IC95% = [-8.9 ; -1.7]

Comparaison : verdict_original vs verdict_anonyme
  b (real->fake) = 19, c (fake->real) = 1, changements = 20
  McNemar : p = 0.0001439 -> significatif (alpha=0.016666666666666666)
  Proportion 'real' : 56.7% -> 44.7%
  Difference = -12.0 points, IC95% = [-17.5 ; -6.5]

Comparaison : verdict_oppose vs verdict_anonyme
  b (real->fake) = 26, c (fake->real) = 0, changements = 26
  McNemar : p = 9.443e-07 -> significatif (alpha=0.016666666666666666)
  Proportion 'real' : 62.0% -> 44.7%
  Difference = -17.3 points, IC95% = [-23.4 ; -11.3]



In [13]:
chemin_in = "/kaggle/input/datasets/imenec/experiences-results/gemma_D_WRAG.csv"
print("Gemma sur catégorie D sans RAG")
# cat D , original = peu connu
# connu vs peu_conu
res1 = comparer_deux_colonnes(chemin_in, "verdict_oppose", "verdict_original")
afficher_resultat(res1)

# peu_connu vs anonyme
res2 = comparer_deux_colonnes(chemin_in, "verdict_original", "verdict_anonyme")
afficher_resultat(res2)

# connu vs anonyme
res3 = comparer_deux_colonnes(chemin_in, "verdict_oppose", "verdict_anonyme")
afficher_resultat(res3)

Gemma sur catégorie D sans RAG
Comparaison : verdict_oppose vs verdict_original
  b (real->fake) = 5, c (fake->real) = 0, changements = 5
  McNemar : p = 0.07364 -> non significatif (alpha=0.016666666666666666)
  Proportion 'real' : 70.7% -> 67.3%
  Difference = -3.3 points, IC95% = [-6.2 ; -0.5]

Comparaison : verdict_original vs verdict_anonyme
  b (real->fake) = 13, c (fake->real) = 4, changements = 17
  McNemar : p = 0.05235 -> non significatif (alpha=0.016666666666666666)
  Proportion 'real' : 67.3% -> 61.3%
  Difference = -6.0 points, IC95% = [-11.3 ; -0.7]

Comparaison : verdict_oppose vs verdict_anonyme
  b (real->fake) = 15, c (fake->real) = 1, changements = 16
  McNemar : p = 0.001154 -> significatif (alpha=0.016666666666666666)
  Proportion 'real' : 70.7% -> 61.3%
  Difference = -9.3 points, IC95% = [-14.3 ; -4.3]



In [15]:
chemin_in = "/kaggle/input/datasets/imenec/experiences-results/qwen_B_RAG.csv"
print("Qwen sur catégorie B avec RAG")
# cat B , original = connu
# connu vs peu_conu
res1 = comparer_deux_colonnes(chemin_in, "verdict_original", "verdict_oppose")
afficher_resultat(res1)

# peu_connu vs anonyme
res2 = comparer_deux_colonnes(chemin_in, "verdict_oppose", "verdict_anonyme")
afficher_resultat(res2)

# connu vs anonyme
res3 = comparer_deux_colonnes(chemin_in, "verdict_original", "verdict_anonyme")
afficher_resultat(res3)

Qwen sur catégorie B avec RAG
Comparaison : verdict_original vs verdict_oppose
  b (real->fake) = 3, c (fake->real) = 3, changements = 6
  McNemar : p = 0.6831 -> non significatif (alpha=0.016666666666666666)
  Proportion 'real' : 49.3% -> 49.3%
  Difference = +0.0 points, IC95% = [-3.2 ; +3.2]

Comparaison : verdict_oppose vs verdict_anonyme
  b (real->fake) = 3, c (fake->real) = 2, changements = 5
  McNemar : p = 1 -> non significatif (alpha=0.016666666666666666)
  Proportion 'real' : 49.3% -> 48.7%
  Difference = -0.7 points, IC95% = [-3.6 ; +2.3]

Comparaison : verdict_original vs verdict_anonyme
  b (real->fake) = 3, c (fake->real) = 2, changements = 5
  McNemar : p = 1 -> non significatif (alpha=0.016666666666666666)
  Proportion 'real' : 49.3% -> 48.7%
  Difference = -0.7 points, IC95% = [-3.6 ; +2.3]



In [16]:
chemin_in = "/kaggle/input/datasets/imenec/experiences-results/qwen_B_WRAG.csv"
print("Qwen sur catégorie B sans RAG")
# cat B , original = connu
# connu vs peu_conu
res1 = comparer_deux_colonnes(chemin_in, "verdict_original", "verdict_oppose")
afficher_resultat(res1)

# peu_connu vs anonyme
res2 = comparer_deux_colonnes(chemin_in, "verdict_oppose", "verdict_anonyme")
afficher_resultat(res2)

# connu vs anonyme
res3 = comparer_deux_colonnes(chemin_in, "verdict_original", "verdict_anonyme")
afficher_resultat(res3)

Qwen sur catégorie B sans RAG
Comparaison : verdict_original vs verdict_oppose
  b (real->fake) = 16, c (fake->real) = 7, changements = 23
  McNemar : p = 0.09529 -> non significatif (alpha=0.016666666666666666)
  Proportion 'real' : 48.7% -> 42.7%
  Difference = -6.0 points, IC95% = [-12.2 ; +0.2]

Comparaison : verdict_oppose vs verdict_anonyme
  b (real->fake) = 30, c (fake->real) = 4, changements = 34
  McNemar : p = 1.807e-05 -> significatif (alpha=0.016666666666666666)
  Proportion 'real' : 42.7% -> 25.3%
  Difference = -17.3 points, IC95% = [-24.4 ; -10.2]

Comparaison : verdict_original vs verdict_anonyme
  b (real->fake) = 35, c (fake->real) = 0, changements = 35
  McNemar : p = 9.081e-09 -> significatif (alpha=0.016666666666666666)
  Proportion 'real' : 48.7% -> 25.3%
  Difference = -23.3 points, IC95% = [-30.1 ; -16.6]



In [17]:
chemin_in = "/kaggle/input/datasets/imenec/experiences-results/qwen_D_RAG.csv"
print("Qwen sur catégorie D avec RAG")
# cat D , original = peu connu
# connu vs peu_conu
res1 = comparer_deux_colonnes(chemin_in, "verdict_oppose", "verdict_original")
afficher_resultat(res1)

# peu_connu vs anonyme
res2 = comparer_deux_colonnes(chemin_in, "verdict_original", "verdict_anonyme")
afficher_resultat(res2)

# connu vs anonyme
res3 = comparer_deux_colonnes(chemin_in, "verdict_oppose", "verdict_anonyme")
afficher_resultat(res3)

Qwen sur catégorie D avec RAG
Comparaison : verdict_oppose vs verdict_original
  b (real->fake) = 5, c (fake->real) = 0, changements = 5
  McNemar : p = 0.07364 -> non significatif (alpha=0.016666666666666666)
  Proportion 'real' : 28.7% -> 25.3%
  Difference = -3.3 points, IC95% = [-6.2 ; -0.5]

Comparaison : verdict_original vs verdict_anonyme
  b (real->fake) = 3, c (fake->real) = 2, changements = 5
  McNemar : p = 1 -> non significatif (alpha=0.016666666666666666)
  Proportion 'real' : 25.3% -> 24.7%
  Difference = -0.7 points, IC95% = [-3.6 ; +2.3]

Comparaison : verdict_oppose vs verdict_anonyme
  b (real->fake) = 7, c (fake->real) = 1, changements = 8
  McNemar : p = 0.0771 -> non significatif (alpha=0.016666666666666666)
  Proportion 'real' : 28.7% -> 24.7%
  Difference = -4.0 points, IC95% = [-7.6 ; -0.4]



In [18]:
chemin_in = "/kaggle/input/datasets/imenec/experiences-results/qwen_D_WRAG.csv"
print("Qwen sur catégorie D sans RAG")
# cat D , original = peu connu
# connu vs peu_conu
res1 = comparer_deux_colonnes(chemin_in, "verdict_oppose", "verdict_original")
afficher_resultat(res1)

# peu_connu vs anonyme
res2 = comparer_deux_colonnes(chemin_in, "verdict_original", "verdict_anonyme")
afficher_resultat(res2)

# connu vs anonyme
res3 = comparer_deux_colonnes(chemin_in, "verdict_oppose", "verdict_anonyme")
afficher_resultat(res3)

Qwen sur catégorie D sans RAG
Comparaison : verdict_oppose vs verdict_original
  b (real->fake) = 5, c (fake->real) = 5, changements = 10
  McNemar : p = 0.7518 -> non significatif (alpha=0.016666666666666666)
  Proportion 'real' : 28.0% -> 28.0%
  Difference = +0.0 points, IC95% = [-4.1 ; +4.1]

Comparaison : verdict_original vs verdict_anonyme
  b (real->fake) = 27, c (fake->real) = 0, changements = 27
  McNemar : p = 5.624e-07 -> significatif (alpha=0.016666666666666666)
  Proportion 'real' : 28.0% -> 10.0%
  Difference = -18.0 points, IC95% = [-24.1 ; -11.9]

Comparaison : verdict_oppose vs verdict_anonyme
  b (real->fake) = 27, c (fake->real) = 0, changements = 27
  McNemar : p = 5.624e-07 -> significatif (alpha=0.016666666666666666)
  Proportion 'real' : 28.0% -> 10.0%
  Difference = -18.0 points, IC95% = [-24.1 ; -11.9]

